# Get Sample Data

In [ ]:
import os
import random
import pandas as pd
from sklearn.model_selection import train_test_split

def sample_txt_files_with_subdir(base_dir, samples_per_subdir=50, output_excel='sampled_sentiment_data.xlsx'):

    file_name = 'train_all.xlsx'
    file_path = os.path.join(base_dir, file_name)

    data = pd.read_excel(file_path)

    data['sentiment'] = data['sentiment'].str.title()

    data_neg = data[data['sentiment']=='Negative']
    data_pos = data[data['sentiment']=='Positive']
    data_neu = data[data['sentiment']=='Neutral']

    remove_neg, sample_neg = train_test_split(data_neg,
                                              test_size = 167,
                                              random_state = 42)
    remove_pos, sample_pos = train_test_split(data_pos,
                                              test_size = 167,
                                              random_state = 42)
    remove_neu, sample_neu = train_test_split(data_neu,
                                              test_size = 167,
                                              random_state = 42)

    sample_data = pd.concat([sample_neg, sample_pos, sample_neu])
    print(sample_data.shape)

    output_path = os.path.join(base_dir, output_excel)
    sample_data.to_excel(output_path, index=False)
    print(f"Saved {len(sample_data)} records to {output_excel}")

# Example usage
directory = 'content'
sample_txt_files_with_subdir(directory)


(501, 3)
Saved 501 records to sampled_sentiment_data.xlsx


# English Prompt

## Zero Shot

In [ ]:
sa = pd.read_excel('sampled_sentiment_data.xlsx')
sa = sa[['Text']]

prompt = 'You are an AI assistant specialized in sentiment analysis. Classify the sentiment of the following sentence based on its emotional tone. Choose only one sentiment between: Positive, Negative, or Neutral.'

zero_Shot = prompt + '\n Sentence: \n' + sa['Text'] + '\n Predicted Sentiment:'
zs = pd.DataFrame()
zs['prompt'] = zero_Shot
zs.to_excel('Sentiment Analysis Zero Shot.xlsx', index = False)

## Few Shot

In [ ]:
prompt_fs = '''You are an AI assistant specialized in sentiment analysis. Classify the sentiment of the following sentence based on its emotional tone. Choose only one sentiment between: Positive, Negative, or Neutral.

Example 1:
Sentence:
كل فرحة تصنعها لغيرك ستعود لك بشكل اجمل صباح الخير ..

Sentiment: Positive

Example 2:
Sentence:
سئمت رؤيتكِ في كل أغنية أسمعها في كل شعر أقرأه في منتصف قهوتي في كل خطوة أخطيها في كل دمعة سئمت

Sentiment: Negative

Example 3:
Sentence:
اذا لم تستطع أن تترك اثرا جميلا في القلوب فلا تزرع فيها آلما لا ينسى.

Sentiment: Neutral

The sentence you need to classify
'''

few_Shot = prompt_fs + 'Sentence: \n' + sa['Text'] + '\n Predicted Sentiment:'
fs = pd.DataFrame()
fs['prompt'] = few_Shot
fs.to_excel('Sentiment Analysis Few Shot.xlsx', index = False)

## CoT

In [ ]:
prompt_cot = '''You are an AI assistant specialized in sentiment analysis. Classify the sentiment of the following sentence based on its emotional tone. Choose only one sentiment between: Positive, Negative, or Neutral.

Step 1: Read the sentence
Carefully read the sentence to fully understand its meaning, context, and tone. Consider both explicit statements and any implied emotional cues.

Step 2: Identify Emotionally Charged Language
- Highlight positive language, such as words indicating satisfaction, happiness, or praise (e.g., great, amazing, love, well-done).
- Highlight negative language, such as words indicating dissatisfaction, frustration, or criticism (e.g., terrible, hate, broken, disappointing).
- Neutral statements neither praise nor criticize.

Step 3: Analyze the Emotional Balance
Consider the overall tone and intent of the sentence, including sarcasm or contrast.

Step 4: Determine Sentiment
- If positive sentiment dominates, classify as Positive.
- If negative sentiment dominates, classify as Negative.
- If there is no clear emotional direction or the content is purely factual, classify as Neutral.

Example 1:
كل فرحة تصنعها لغيرك ستعود لك بشكل اجمل صباح الخير ..

Thoughts:
- Positive words: فرحة, اجمل, الخير
- Negative words: None
- Emotional Balance: tweet uses only positive language and promotes an uplifting message about generosity and the return of happiness.
- Sentiment: positive

Predicted Sentiment: Positive

Example 2:
سئمت رؤيتكِ في كل أغنية أسمعها في كل شعر أقرأه في منتصف قهوتي في كل خطوة أخطيها في كل دمعة سئمت


Thoughts:
- Positive word: None
- Negative words: سئمت, دمعة
- Emotional Balance: tweet uses only negative language which revolves around fatigue, sadness, and being overwhelmed by memories.
- Sentiment: Negative

Predicted Sentiment: Negative

Example 3:
اذا لم تستطع أن تترك اثرا جميلا في القلوب فلا تزرع فيها آلما لا ينسى.

Thoughts:
- Positive words: جميلاً
- Negative words: ألماً
- Emotional Balance: tweet mentions both positive and negative outcomes, the overall tone is cautionary and moralistic, not emotionally expressive.
- Sentiment: Neutral

Predicted Sentiment: Neutral

The sentence you need to classify
'''

cot = prompt_cot + 'Sentence: \n' + sa['Text']
CoT = pd.DataFrame()
CoT['prompt'] = cot
CoT.to_excel('Sentiment Analysis CoT.xlsx', index = False)

# Arabic Prompts

For Arabic models like Jais

## Zero Shot

In [ ]:
prompt = 'أنت مساعد ذكاء اصطناعي متخصص في تحليل المشاعر. صنّف مشاعر الجملة التالية بناءً على نبرتها العاطفية. اختر شعورًا واحدًا فقط من بين: إيجابي، سلبي، أو محايد.'

zero_Shot = prompt + 'الجملة: \n' + sa['Text'] + '\n المشاعر المتوقعة:'
zs = pd.DataFrame()
zs['prompt'] = zero_Shot
zs.to_excel('Setntiment Analysis Jais Arabic Shot.xlsx', index = False)

## Few Shot

In [ ]:
prompt_fs = '''أنت مساعد ذكاء اصطناعي متخصص في تحليل المشاعر. صنّف مشاعر الجملة التالية بناءً على نبرتها العاطفية. اختر شعورًا واحدًا فقط من بين: إيجابي، سلبي، أو محايد.

مثال 1:
الجملة:
كل فرحة تصنعها لغيرك ستعود لك بشكل اجمل صباح الخير ..

المشاعر المتوقعة: إيجابي

مثال 2:
الجملة:
سئمت رؤيتكِ في كل أغنية أسمعها في كل شعر أقرأه في منتصف قهوتي في كل خطوة أخطيها في كل دمعة سئمت

المشاعر المتوقعة: سلبي

مثال 3:
الجملة:
اذا لم تستطع أن تترك اثرا جميلا في القلوب فلا تزرع فيها آلما لا ينسى.

المشاعر المتوقعة: محايد

الجملة الذي عليك تصنيفها
'''

few_Shot = prompt_fs + 'الجملة: \n' + sa['Text'] + '\n المشاعر المتوقعة:'
fs = pd.DataFrame()
fs['prompt'] = few_Shot
fs.to_excel('Sentiment Analysis Jais Arabic Shot.xlsx', index = False)

## CoT

In [ ]:
prompt_cot = '''أنت مساعد ذكاء اصطناعي متخصص في تحليل المشاعر. صنّف مشاعر الجملة التالية بناءً على نبرتها العاطفية. اختر شعورًا واحدًا فقط من بين: إيجابي، سلبي، أو محايد.

الخطوة 1: قراءة الجملة
اقرأ الجملة بعناية لفهم معناها الكامل وسياقها ونبرتها. خذ بعين الاعتبار التصريحات الصريحة وأي إشارات عاطفية ضمنية.

الخطوة 2: تحديد اللغة العاطفية
- قم بتمييز الكلمات الإيجابية، مثل الكلمات التي تدل على الرضا أو السعادة أو المدح (مثل: رائع، مدهش، أحب، أحسنت).
- قم بتمييز الكلمات السلبية، مثل الكلمات التي تدل على عدم الرضا أو الإحباط أو النقد (مثل: سيء، أكره، مكسور، مخيب).
- العبارات المحايدة لا تتضمن مدحًا أو نقدًا.

الخطوة 3: تحليل التوازن العاطفي
انظر إلى النبرة العامة ونية الجملة، بما في ذلك السخرية أو التناقض إن وجد.

الخطوة 4: تحديد الشعور
- إذا طغت المشاعر الإيجابية، صنفها كـ "إيجابية".
- إذا طغت المشاعر السلبية، صنفها كـ "سلبية".
- إذا لم يكن هناك اتجاه عاطفي واضح أو كانت الجملة معلوماتية بحتة، صنفها كـ "محايدة".

مثال 1:
الجملة:
كل فرحة تصنعها لغيرك ستعود لك بشكل اجمل صباح الخير ..

التحليل:
الكلمات الإيجابية: فرحة، أجمل، الخير
الكلمات السلبية: لا يوجد
التوازن العاطفي: التغريدة تستخدم لغة إيجابية فقط وتروج لرسالة مفعمة بالتفاؤل والعطاء.
الشعور: إيجابي

المشاعر المتوقعة: إيجابي

مثال 2:
الجملة:
سئمت رؤيتكِ في كل أغنية أسمعها في كل شعر أقرأه في منتصف قهوتي في كل خطوة أخطيها في كل دمعة سئمت

التحليل:
الكلمات الإيجابية: لا يوجد
الكلمات السلبية: سئمت، دمعة
التوازن العاطفي: التغريدة تحتوي فقط على كلمات سلبية تدور حول التعب والحزن والاجتياح العاطفي.
الشعور: سلبي

المشاعر المتوقعة: سلبي

مثال 3:
الجملة:
إذا لم تستطع أن تترك أثراً جميلاً في القلوب فلا تزرع فيها ألماً لا يُنسى.

التحليل:
الكلمات الإيجابية: جميلاً
الكلمات السلبية: ألماً
التوازن العاطفي: التغريدة تحتوي على إشارات إيجابية وسلبية في آن واحد، ونبرتها العامة وعظية وتحذيرية دون تعبير عاطفي مباشر.
الشعور: محايد

المشاعر المتوقعة: محايد

الجملة الذي عليك تصنيفها
'''

cot = prompt_cot + 'الجملة: \n' + sa['Text']
CoT = pd.DataFrame()
CoT['prompt'] = cot
CoT.to_excel('Sentiment Analysis Arabic CoT.xlsx', index = False)